<img src="../../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

# 00 · ALCHEMI Core Playbook

> **TECHNICALLY VALIDATED, RENDERED-REVIEWED DRAFT — HUMAN CELL REVIEW REQUIRED**

**Goal:** Follow the small set of Toolkit objects that carry an atomistic calculation from structures to models and simulation workflows.

**You will:**

- convert molecular structures to `AtomicData`;
- pack variable-size systems into a `Batch`;
- evaluate a model and inspect its outputs;
- add hooks, relaxation, and molecular dynamics; and
- preview training and distributed execution.

**Time to complete: about 90 minutes.**

## Where NVIDIA ALCHEMI fits

[ALCHEMI](https://developer.nvidia.com/cuda/cuda-x-libraries/alchemi) brings together Python building blocks, accelerated kernels, and deployable services for atomistic workflows.

- **ALCHEMI Toolkit** is a GPU-first Python framework with a unified, composable API for MLIPs and custom models. It provides GPU-native atomic data and batching, model adapters, MD classes, hooks, model training, and single- to multi-GPU pipelines.  
[GitHub repo](https://github.com/NVIDIA/nvalchemi-toolkit) · [Docs](https://nvidia.github.io/nvalchemi-toolkit/) · Apache 2.0 license

- **Toolkit-Ops** supplies GPU-optimized, batched operations for neighbor lists, dynamics, dispersion, and electrostatics. The next section shows how its framework bindings connect to lower-level implementations.  
[GitHub repo](https://github.com/NVIDIA/nvalchemi-toolkit-ops) · [Docs](https://nvidia.github.io/nvalchemi-toolkit-ops/) · Apache 2.0 license

- **ALCHEMI NIM microservices** package supported atomistic workflows as cloud-ready services. The current catalog includes Batched Geometry Relaxation for structural optimization and Batched Molecular Dynamics. Self-hosting uses an NVIDIA AI Enterprise license.  
[Transparency card](https://docs.nvidia.com/nim/alchemi/alchemi-bgr/1.0.0/ai-transparency-card/overview.html) · [Docs](https://docs.nvidia.com/nim/alchemi/alchemi-bgr/latest/index.html)

## Explore the Toolkit capability map

Toolkit combines four areas of work. Hover or focus a card to see what it represents, where it is useful, and where to continue. The focused course notebooks are in progress.

In [88]:
from helpers import core as helpers

helpers.show_capability_map()

<details>
<summary>Open documentation without hover</summary>

The focused course notebooks are still in progress.

- **Data and state:** [documentation](https://nvidia.github.io/nvalchemi-toolkit/userguide/data.html) · <span style="color:#7C8794;">Part 01 · in progress</span> · <span style="color:#7C8794;">Part 02 · in progress</span>
- **Models and potentials:** [documentation](https://nvidia.github.io/nvalchemi-toolkit/userguide/models.html) · <span style="color:#7C8794;">Part 03 · in progress</span>
- **Simulation workflows:** [documentation](https://nvidia.github.io/nvalchemi-toolkit/userguide/dynamics.html) · <span style="color:#7C8794;">Part 04 · in progress</span> · <span style="color:#7C8794;">Part 05 · in progress</span> · <span style="color:#7C8794;">Part 06 · in progress</span>
- **Training and scale:** [training documentation](https://nvidia.github.io/nvalchemi-toolkit/userguide/training.html) · [distributed documentation](https://nvidia.github.io/nvalchemi-toolkit/userguide/distributed.html) · <span style="color:#7C8794;">Part 07 · in progress</span> · <span style="color:#7C8794;">Part 08 · in progress</span>

Browse the [official Toolkit examples](https://nvidia.github.io/nvalchemi-toolkit/examples/) for complete applications built from the same public objects.

</details>

## Before you run

The molecular sections use ASE structures, Toolkit data, AIMNet2 predictions, and bounded FIRE2 updates. A focused MatterViz view keeps the original ASE structure available for spatial inspection while the Toolkit tensors stay on the selected device.

The molecular-dynamics section uses periodic Lennard-Jones argon because that compact model supplies the complete energy and force calculation. The training section uses generated four-atom systems so the optimizer steps finish quickly. Each section rebuilds the inputs needed for its own API.

CUDA is the course target. CPU mode supports a quick API walkthrough, with slower model and dynamics cells.

In [70]:
import shutil
import tempfile
from pathlib import Path

import pandas as pd
import torch
from ase import Atoms
from helpers import core as helpers
from nvalchemi.data import (
    AtomicData,
    AtomicDataZarrReader,
    AtomicDataZarrWriter,
    Batch,
    DataLoader,
    Dataset,
)
from nvalchemi.distributed import DistributedManager, DomainConfig, DomainParallel
from nvalchemi.dynamics import (
    FIRE2,
    NVE,
    BaseDynamics,
    ConvergenceHook,
    DynamicsStage,
    HostMemory,
)
from nvalchemi.dynamics.hooks import NaNDetectorHook, SnapshotHook
from nvalchemi.hooks import DynamicsContext, WrapPeriodicHook
from nvalchemi.models.aimnet2 import AIMNet2Wrapper
from nvalchemi.models.base import BaseModelMixin, ModelConfig
from nvalchemi.models.lj import LennardJonesModelWrapper
from nvalchemi.neighbors import compute_neighbors
from nvalchemi.training import (
    EnergyMSELoss,
    FineTuningStrategy,
    OptimizerConfig,
    default_training_fn,
)

In [71]:
helpers.configure_presentation()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

device_label = "CUDA GPU" if device.type == "cuda" else "CPU"
print(f"Compute device: {device_label}")

Compute device: CUDA GPU


<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 01 · Build the molecule with ASE

How can the same four-atom molecule become a validated Toolkit graph whether our source is an ASE object or PyTorch tensors?

We start with ethyne because its two C–H bonds and C≡C axis make the molecule-to-graph transition easy to inspect. Coordinates are in ångström, from a curated [NCI Atlas](https://github.com/Honza-R/NCIAtlas) subset ([CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)).

`AtomicData` tensors and later model execution use the selected compute device. The ASE object remains an ordinary in-memory structure; this does not place Zarr storage on that device.

In [72]:
ethyne = Atoms(
    symbols=["C", "H", "C", "H"],
    positions=[
        [-2.011548349, 0.283130903, -0.000000131],
        [-1.057741761, 0.762468607, -0.000000367],
        [-3.070459929, -0.274876730, 0.000000058],
        [-4.013236394, -0.761855986, 0.000000226],
    ],
    pbc=False,
)
ethyne.info["charge"] = 0

Select the explicitly constructed `ethyne` object and inspect its identity. The geometry comes from the curated [NCI Atlas](https://github.com/Honza-R/NCIAtlas) subset ([CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)).

In [73]:
print(ethyne)
print(
    f"{ethyne.get_chemical_formula()} · {len(ethyne)} atoms · "
    f"net charge {ethyne.info['charge']:+d} e · positions in Å"
)

Atoms(symbols='CHCH', pbc=False)
C2H2 · 4 atoms · net charge +0 e · positions in Å


## View the ASE molecule

MatterViz draws bonds from ASE covalent radii. The small, linear molecule makes missing or occluded bond geometry immediately visible.

In [87]:
helpers.show_molecule(ethyne, bonds=helpers.infer_bonds(ethyne), height=360)

## Convert through the public ASE route

Use `AtomicData.from_atoms(...)` when a chemistry library, file reader, or simulation already provides an ASE `Atoms` object. Toolkit validates the mapped fields and places its tensors on the requested compute device.

In [75]:
ethyne_data = AtomicData.from_atoms(ethyne, device=device)

In [80]:
property_groups = {
    "node": set(ethyne_data.node_properties),
    "edge": set(ethyne_data.edge_properties),
    "system": set(ethyne_data.system_properties),
}
field_level = {
    name: level for level, names in property_groups.items() for name in names
}
for name in ("atomic_numbers", "positions", "charge"):
    tensor = getattr(ethyne_data, name)
    print(
        f"{name:15} {field_level[name]:7} {tuple(tensor.shape)!s:8} "
        f"{tensor.dtype} {tensor.device}"
    )

atomic_numbers  node    (4,)     torch.int32 cuda:0
positions       node    (4, 3)   torch.float32 cuda:0
charge          system  (1, 1)   torch.float32 cuda:0


## Construct AtomicData directly from tensors

Use the public constructor when a reader or simulation already supplies tensors. We will describe the same ethyne graph with explicit node tensors and its verified system charge.

In [77]:
atomic_numbers = torch.tensor([6, 1, 6, 1], dtype=torch.int32, device=device)
positions = torch.tensor(
    [
        [-2.011548349, 0.283130903, -0.000000131],
        [-1.057741761, 0.762468607, -0.000000367],
        [-3.070459929, -0.274876730, 0.000000058],
        [-4.013236394, -0.761855986, 0.000000226],
    ], dtype=torch.float32, device=device, )  # Cartesian coordinates in Å

charge = torch.tensor([[0.0]], dtype=torch.float32, device=device)

In [ ]:
direct_ethyne_data = AtomicData(
    atomic_numbers=atomic_numbers,
    positions=positions,
    charge=charge,
)

## Compare the two construction routes

`AtomicData` equality compares the chemical hash, so one expression answers whether both routes built the same structure. The ASE converter also fills optional node fields such as masses; direct construction carries only the fields we passed.

In [82]:
ethyne_data == direct_ethyne_data

True

## Verify and render a real anion

Use benzoate from the curated [NCI Atlas](https://github.com/Honza-R/NCIAtlas) subset ([CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)). Its source metadata and `AtomicData` system field both report net charge −1 e.

MatterViz renders atoms and explicit bonds. The compact badge reads `AtomicData.charge`; it is not a native scene annotation and does not imply an atom-by-atom charge distribution.

In [83]:
benzoate, benzoate_record = helpers.load_benzoate_anion()
benzoate_data = AtomicData.from_atoms(benzoate, device=device)
{
    "identity": (benzoate_record["formula"], len(benzoate)),
    "source charge [e]": benzoate_record["charge"],
    "AtomicData charge [e]": float(benzoate_data.charge.item()),
}

{'identity': ('C7H5O2', 14),
 'source charge [e]': -1,
 'AtomicData charge [e]': -1.0}

In [ ]:
helpers.charge_badge(benzoate_data)

Infer benzoate connectivity with the same public ASE method, then pass those bonds to MatterViz.

In [ ]:
benzoate_data

In [ ]:
helpers.show_molecule(benzoate, bonds=helpers.infer_bonds(benzoate), height=360)

In [ ]:
{
    "charge provenance": benzoate_record["source"],
    "verified net charge [e]": float(benzoate_data.charge.item()),
}

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 01 · Pack several molecules into one batch

Load ethyne, phenol, and 2,3-dimethylbutane from the curated [NCI Atlas](https://github.com/Honza-R/NCIAtlas) subset ([CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)). Their different atom counts make graph boundaries visible.

Convert and inspect the raw `Batch` before adding any API-specific input scaffolding.

In [ ]:
labels = ("Ethyne", "Phenol", "2,3-dimethylbutane")
atoms, molecule_records = helpers.load_molecules(labels)
for record in molecule_records:
    print(
        f"{record['label']:20} {record['formula']:6} {record['atoms']:3} atoms  "
        f"charge {record['charge']:+d} e  {record['source']}"
    )

In [ ]:
molecule_data = [
    AtomicData.from_atoms(molecule, device=device) for molecule in atoms
]

In [ ]:
example_batch = Batch.from_data_list(molecule_data, device=device)
example_batch.num_graphs, example_batch.num_nodes, example_batch.device

In [ ]:
{
    "graphs / atoms": (example_batch.num_graphs, example_batch.num_nodes),
    "node fields": sorted(example_batch.keys["node"]),
    "system fields": sorted(example_batch.keys["system"]),
    "has energy": "energy" in example_batch,
    "has forces": "forces" in example_batch,
}

In [ ]:
# Input metadata only; these are not model predictions.
example_batch.add_key(
    "reference_temperature_K",
    [torch.tensor([[value]], device=device) for value in (250.0, 300.0, 350.0)],
    level="system",
)

In [ ]:
{
    "registered": "reference_temperature_K" in example_batch,
    "shape": tuple(example_batch.reference_temperature_K.shape),
    "values [K]": example_batch.reference_temperature_K.flatten().tolist(),
}

### Try it: predict the packed boundaries

`num_nodes_per_graph` stores each molecule's atom count. `batch_ptr` gives the start and stop offsets, and `batch_idx` maps every packed atom row to its source graph.

Add the three atom counts and predict the final value of `batch_ptr`. It must equal the total number of packed atom rows.

In [ ]:
example_counts = example_batch.num_nodes_per_graph.tolist()
example_counts

In [ ]:
example_ptr = example_batch.batch_ptr.tolist()
example_ptr

In [ ]:
example_batch.batch_idx

In [ ]:
helpers.plot_batch_ownership(example_batch, labels)

<details>
<summary>Check the answer</summary>

The three molecules contain 4, 13, and 20 atoms. Their cumulative boundaries are `[0, 4, 17, 37]`, so the final pointer equals the 37 packed atom rows.

</details>

### Try it: recover one molecule

Set `graph_index` to 0, 1, or 2. `get_data(...)` reconstructs that molecule as one `AtomicData` object from the stored boundaries.

In [ ]:
graph_index = 1
recovered_graph = example_batch.get_data(graph_index)
labels[graph_index], recovered_graph.num_nodes, recovered_graph.device

<details>
<summary>Check the answer</summary>

Indexes 0, 1, and 2 recover ethyne, phenol, and 2,3-dimethylbutane with 4, 13, and 20 atoms. The recovered `AtomicData` object stays on the batch device.

</details>

**Go deeper:** [AtomicData and Batch](../01-atomicdata-batch/atomicdata-and-batch.ipynb) continues with indexing, round trips, custom fields, neighbor construction, and advanced buffering patterns.

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 02 · Save records and recover the next model input

The three molecular graphs currently live only in memory. Zarr stores each `AtomicData` record, including its atom-level fields, system-level fields, and the `record_id` added below. When another notebook or training job loads those records into a `Batch`, Toolkit reconstructs the graph boundaries during collation. The original ASE objects are no longer needed.

This sequence makes CPU copies, saves them to a temporary filesystem-backed Zarr store, releases the source objects, reopens the saved records as CPU tensors, and collates all three through `Dataset` and `DataLoader`.

<div style="display:block;box-sizing:border-box;width:100%;max-width:100%;min-width:0;background:#151A1F;color:#F3F4F6;border:1px solid #30363D;border-radius:8px;padding:0.82rem 0.95rem;line-height:1.45;overflow:hidden;overflow-wrap:anywhere;">
  <div style="color:#76B900;font-size:0.76rem;font-weight:700;letter-spacing:0.06em;margin-bottom:0.3rem;">ALCHEMI TOOLKIT API</div>
  <code style="display:block;color:#FFFFFF;font-size:1rem;font-weight:650;white-space:normal;overflow-wrap:anywhere;">AtomicDataZarrWriter → AtomicDataZarrReader → Dataset → DataLoader</code>
  <div style="color:#CDD2D8;margin-top:0.5rem;font-size:0.92rem;">Save graph fields, reopen CPU tensors, restore validated <code>AtomicData</code>, and collate a <code>Batch</code> on the dataset target device.</div>
</div>

<img src="assets/zarr-data-flow-71a1aace19612fd3.svg" alt="Records are saved once to a Zarr store on disk or in CPU storage, loaded as CPU tensors by Reader, moved by Dataset to its target device, batched and prefetched by DataLoader, and emitted as a Batch on the model device." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.8rem 0 0.45rem;">

*Records are saved once, then loaded and batched for model use.*

In [ ]:
records_to_save = [
    example_batch.get_data(index).clone().to("cpu")
    for index in range(example_batch.num_graphs)
]

In [ ]:
RECORD_IDS = (101, 102, 103)
for record_id, record in zip(RECORD_IDS, records_to_save, strict=True):
    # record_id belongs to the complete graph and survives storage as a system field.
    record.add_system_property(
        "record_id", torch.tensor([record_id], dtype=torch.int64)
    )

In [ ]:
for record, label in zip(records_to_save, labels, strict=True):
    print(
        f"record_id {int(record.record_id.item()):4} {label:20} "
        f"{record.num_nodes:3} atoms  charge {float(record.charge.item()):+g} e  "
        f"{record.device}"
    )

### Write a self-describing graph store

`AtomicDataZarrWriter` records concatenated atom fields, graph pointers, and field ownership. The temporary path keeps this walkthrough isolated from project data.

First construct the writer and the CPU `Batch`. Then inspect graph count, boundaries, and stable IDs before writing.

In [ ]:
STORE = Path(tempfile.mkdtemp(prefix="alchemi-core-zarr-")) / "molecules.zarr"
writer = AtomicDataZarrWriter(STORE)

In [ ]:
source_batch = Batch.from_data_list(records_to_save, device="cpu")

In [ ]:
expected_round_trip = {
    "record IDs": source_batch.record_id.tolist(),
    "boundaries": source_batch.batch_ptr.tolist(),
    "atomic numbers": source_batch.atomic_numbers.clone(),
    "positions [Å]": source_batch.positions.clone(),
    "temperatures [K]": source_batch.reference_temperature_K.clone(),
}
{
    "store": STORE.name,
    "graphs / atoms": (source_batch.num_graphs, source_batch.num_nodes),
    "record IDs": expected_round_trip["record IDs"],
    "boundaries": expected_round_trip["boundaries"],
}

In [ ]:
writer.write(source_batch)

In [ ]:
del writer, source_batch, records_to_save
source_objects_released = all(
    name not in globals() for name in ("writer", "source_batch", "records_to_save")
)
source_objects_released

### Reopen the store through the public reader

The source `AtomicData` objects and source `Batch` are gone. `AtomicDataZarrReader` now owns random-access I/O. It returns raw CPU tensor dictionaries plus metadata; `reader.field_levels` records whether each field belongs to atoms or complete systems.

In [ ]:
reader = AtomicDataZarrReader(STORE)
len(reader)

Load the saved records together to inspect stable identity and the stored schema before selecting one.

In [ ]:
loaded_records = reader.read_many(range(len(reader)))
{
    "records": len(loaded_records),
    "record IDs": [int(raw["record_id"].item()) for raw, _ in loaded_records],
    "atom fields": sorted(
        name for name, level in reader.field_levels.items() if level == "atom"
    ),
    "system fields": sorted(
        name for name, level in reader.field_levels.items() if level == "system"
    ),
}

### Compare one loaded record with its source values

`reader.read(1)` loads the second saved record as CPU tensors and separate metadata. Compare identity, positions in ångström, atomic numbers, and shapes against the tensor snapshot retained before the source objects were released.

In [ ]:
raw_record, raw_metadata = reader.read(1)
raw_metadata

In [ ]:
second_start, second_stop = expected_round_trip["boundaries"][1:3]
{
    "record ID": int(raw_record["record_id"].item()),
    "positions shape": tuple(raw_record["positions"].shape),
    "positions unit": "Å",
    "atomic numbers match": torch.equal(
        raw_record["atomic_numbers"],
        expected_round_trip["atomic numbers"][second_start:second_stop],
    ),
    "positions match": torch.equal(
        raw_record["positions"],
        expected_round_trip["positions [Å]"][second_start:second_stop],
    ),
    "metadata": raw_metadata,
}

### Restore validated graph objects with `Dataset`

`Dataset` validates reader output as `AtomicData` and moves each requested sample to its target device. Index one item before introducing batching.

In [ ]:
dataset = Dataset(reader, device="cpu", num_workers=2)
len(dataset)

In [ ]:
validated_record, validated_metadata = dataset[1]

In [ ]:
{
    "object": type(validated_record).__name__,
    "record ID": int(validated_record.record_id.item()),
    "atoms": validated_record.num_nodes,
    "positions shape": tuple(validated_record.positions.shape),
    "device": str(validated_record.device),
    "metadata": validated_metadata,
}

<div style="display:block;box-sizing:border-box;width:100%;max-width:100%;min-width:0;background:#F2F3F1;color:#1B1E20;border:1px solid #D6D9D4;border-radius:8px;padding:0.78rem 0.95rem;line-height:1.45;overflow:hidden;overflow-wrap:anywhere;">
  <div style="color:#725B22;font-size:0.78rem;font-weight:700;letter-spacing:0.03em;margin-bottom:0.25rem;">💡 Highlight</div>
  <div style="min-width:0;font-size:0.98rem;overflow-wrap:anywhere;"><strong>Bring another data source.</strong> <code>Dataset</code> is not tied to Zarr. A custom <code>Reader</code> can translate another file format, database, or remote source into the raw CPU tensor records that <code>Dataset</code> validates as <code>AtomicData</code>. It supplies logical length (<code>__len__</code>), field ownership (<code>field_levels</code>), and ordered reads through <code>read_many(...)</code>, usually by implementing <code>_load_sample(...)</code> or <code>_load_many_samples(...)</code>. Once that contract is met, <code>Dataset → DataLoader → Batch</code> stays the same. <a href="../02-zarr-data-loading/zarr-data-loading.ipynb">Part 02</a> implements a real custom Reader.</div>
</div>

### Collate requested records with `DataLoader`

`DataLoader` uses keyword-only batching and prefetch settings. With `shuffle=False`, the emitted graph order should match `RECORD_IDS`. The result is the same graph-aware `Batch` interface used by models and dynamics.

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=3,
    shuffle=False,
    prefetch_factor=1,
    use_streams=False,
)

In [ ]:
first_batch = next(iter(loader))

In [ ]:
{
    "object": type(first_batch).__name__,
    "graphs / atoms": (first_batch.num_graphs, first_batch.num_nodes),
    "record IDs": first_batch.record_id.tolist(),
    "boundaries": first_batch.batch_ptr.tolist(),
    "IDs preserved": first_batch.record_id.tolist() == expected_round_trip["record IDs"],
    "atomic numbers preserved": torch.equal(
        first_batch.atomic_numbers, expected_round_trip["atomic numbers"]
    ),
    "positions preserved": torch.equal(
        first_batch.positions, expected_round_trip["positions [Å]"]
    ),
    "temperatures preserved": torch.equal(
        first_batch.reference_temperature_K,
        expected_round_trip["temperatures [K]"],
    ),
}

In [ ]:
dataset.close()
shutil.rmtree(STORE.parent)
STORE.parent.exists()

Zarr remains on disk or in CPU storage, and `AtomicDataZarrReader` materializes CPU tensors from saved arrays and metadata rather than live Python objects. Here `Dataset(device="cpu")` sets the emitted device; `DataLoader` batches and prefetches without a device argument. [Part 02](../02-zarr-data-loading/zarr-data-loading.ipynb) covers pinned CPU memory, stream transfers, and `InMemoryDataset` cache choices.

**Go deeper:** [Data loading with Zarr](../02-zarr-data-loading/zarr-data-loading.ipynb) adds custom readers, validation boundaries, larger collections, cache choices, and device-aware loading. The [official Zarr trajectory example](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/02_trajectory_zarr_io.html) applies the same persistence path to simulation snapshots.

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 03 · Evaluate a pretrained model

`AIMNet2Wrapper` presents an AIMNet2 checkpoint through the model interface used by Toolkit dynamics. The environment resolves `aimnet2_wb97m_d3_0`, one member of the four-model AIMNet2 wB97M-D3 ensemble, and verifies the local file before loading it.

The model targets closed-shell organic and main-group molecular chemistry over H, B, C, N, O, F, Si, P, S, Cl, As, Se, Br, and I. It is not a transition-metal or bulk-material model. [AIMNetCentral](https://github.com/isayevlab/aimnetcentral/) lists an MIT license; Toolkit is Apache 2.0. This tutorial does not redistribute the checkpoint.

The wrapper call below evaluates the checkpoint base without adding a separate long-range electrostatics or dispersion model. Start by inspecting its required inputs, available outputs, active outputs, device, and neighbor configuration.

In [ ]:
aimnet = AIMNet2Wrapper.from_checkpoint(
    helpers.model_checkpoint(), device=device, compile_model=False
).eval()
aimnet = helpers.freeze_model(aimnet)

In [ ]:
model = aimnet
{
    "adapter": type(model).__name__,
    "device": str(next(model.parameters()).device),
    "input fields": sorted(model.input_data()),
    "available outputs": sorted(model.model_config.outputs),
    "periodic support": model.model_config.supports_pbc,
}

### Choose outputs and prepare neighbors

`set_config("active_outputs", ...)` selects the quantities the wrapper should return. Energy is a system-level output. Forces have one three-component vector per atom.

A model also declares how it expects neighbors. `compute_neighbors(...)` reads that configuration and updates a batch for one model call.

<div style="box-sizing:border-box;width:100%;max-width:100%;background:#151A1F;color:#F3F4F6;border:1px solid #30363D;border-radius:8px;padding:0.72rem 0.9rem;overflow-wrap:anywhere;">
  <div style="color:#76B900;font-size:0.75rem;font-weight:700;letter-spacing:0.05em;">ALCHEMI TOOLKIT API</div>
  <code style="display:block;color:#FFFFFF;font-size:0.94rem;font-weight:650;margin-top:0.28rem;white-space:normal;">compute_neighbors(batch, config=model.model_config.neighbor_config)</code>
  <div style="color:#CDD2D8;font-size:0.9rem;margin-top:0.32rem;">Writes the neighbor representation requested by the model. Iterative workflows use <code>model.make_neighbor_hooks()</code>.</div>
</div>

In [ ]:
model.set_config("active_outputs", {"energy", "forces"})
model.model_config.active_outputs, model.model_config.neighbor_config

In [ ]:
model_graphs = [
    AtomicData.from_atoms(molecule, device=device) for molecule in atoms
]
model_batch = Batch.from_data_list(model_graphs, device=device)
# Outside a dynamics host, compute_neighbors updates the Batch in place.
compute_neighbors(model_batch, config=model.model_config.neighbor_config)
{
    "neighbor matrix": tuple(model_batch.neighbor_matrix.shape),
    "format": model.model_config.neighbor_config.format.name,
}

### Run one batched forward pass

Before running the model, predict the output shapes. Energy has one row per graph. Forces have one three-component row per atom.

Run the next two cells, then compare the returned shapes with `num_graphs` and `num_nodes`.

In [ ]:
model_output = model(model_batch)

In [ ]:
{
    "energy": (
        tuple(model_output["energy"].shape), str(model_output["energy"].dtype),
        "eV", str(model_output["energy"].device),
    ),
    "forces": (
        tuple(model_output["forces"].shape), str(model_output["forces"].dtype),
        "eV/Å", str(model_output["forces"].device),
    ),
    "graph count": model_batch.num_graphs,
    "atom rows": model_batch.num_nodes,
}

<details>
<summary>Check the prediction</summary>

With three graphs and 37 atoms, energy has shape `(3, 1)` and forces have shape `(37, 3)`. Both outputs remain on the selected device.

</details>

### Attach the returned fields

`model(model_batch)` returns a `ModelOutputs` mapping. `Batch.add_key(...)` registers energy at system level and forces at node level. Toolkit dynamics performs this registration during its compute stage; a standalone call makes each step visible.

The next cell attaches the inference values and extracts ethyne so we can inspect its field groups again.

**Scientific scope:** These cells verify adapter execution, output shapes, dtypes, and labeled units for one `aimnet2-wb97m-d3_0` ensemble member. The later argon example uses a self-contained cutoff Lennard-Jones demonstration model.

**Go deeper:** [Model interfaces and composition](../03-model-interfaces-composition/model-interfaces-composition.ipynb) covers configuration, adapters, and multi-component models. The [official models guide](https://nvidia.github.io/nvalchemi-toolkit/userguide/models.html) documents the shared wrapper interface.

In [ ]:
energy_rows = [
    row.detach().to(model_batch.positions.dtype).unsqueeze(0)
    for row in model_output["energy"]
]
force_rows = list(
    model_output["forces"].detach().split(model_batch.num_nodes_per_graph.tolist())
)
model_batch.add_key("energy", energy_rows, level="system")
model_batch.add_key("forces", force_rows, level="node")
evaluated_ethyne = model_batch.get_data(0)
{
    "node fields after neighbors + model": sorted(evaluated_ethyne.node_properties),
    "edge fields after neighbors + model": sorted(evaluated_ethyne.edge_properties),
    "system fields after model": sorted(evaluated_ethyne.system_properties),
}

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 04 · Observe and protect a simulation with hooks

Hooks attach behavior to named points in a workflow. This run uses model-provided neighbor hooks, a numerical check, and a snapshot recorder.

### Try it: change the snapshot cadence

Set `SNAPSHOT_EVERY` to 2 or 4. The registry table will show the exact frequency passed to `SnapshotHook`.

In [ ]:
neighbor_hooks = model.make_neighbor_hooks()

In [ ]:
SNAPSHOT_EVERY = 4
nan_guard = NaNDetectorHook(frequency=1)
snapshot_sink = HostMemory(capacity=16)
snapshot_hook = SnapshotHook(sink=snapshot_sink, frequency=SNAPSHOT_EVERY)
builtin_hooks = [*neighbor_hooks, nan_guard, snapshot_hook]

In [ ]:
pd.DataFrame(
    {
        "hook": [type(hook).__name__ for hook in builtin_hooks],
        "stage": [hook.stage.name for hook in builtin_hooks],
        "frequency": [hook.frequency for hook in builtin_hooks],
    }
)

<details>
<summary>Check the answer</summary>

The `SnapshotHook` row reports the value of `SNAPSHOT_EVERY`. A value of 2 makes the hook eligible at every second matching dynamics stage; a value of 4 records less often.

</details>

### Record energy and force during a run

A custom hook needs a `stage`, a positive `frequency`, and `__call__(ctx, stage)`. `DynamicsStage.AFTER_COMPUTE` runs after the model has written current energies and forces to `ctx.batch`.

`ForceHistoryHook` records one energy and maximum-force value per molecule. A one-step `BaseDynamics` call below exercises the hook through a real workflow context before FIRE2 reuses it.

In [ ]:
class ForceHistoryHook:
    stage = DynamicsStage.AFTER_COMPUTE
    frequency = 1

    def __init__(self):
        self.rows = []

    def __call__(self, ctx: DynamicsContext, stage: DynamicsStage):
        if ctx.batch is None:
            raise RuntimeError("ForceHistoryHook requires a populated Batch")
        pointers = ctx.batch.batch_ptr.tolist()
        for graph, (start, stop) in enumerate(zip(pointers[:-1], pointers[1:], strict=True)):
            forces = ctx.batch.forces[start:stop].detach()
            self.rows.append({
                "step": int(ctx.step_count),
                "graph": graph,
                "energy_ev": float(ctx.batch.energy[graph].detach().sum().cpu()),
                "fmax_ev_per_a": float(forces.norm(dim=1).max().cpu()),
            })

In [ ]:
history_hook = ForceHistoryHook()
# BaseDynamics calls this hook at its declared stage and frequency.
hook_probe = BaseDynamics(
    model=model, n_steps=1, hooks=[*neighbor_hooks, history_hook]
)

In [ ]:
hook_input = example_batch.clone()
hook_input.add_key(
    "energy", [row.detach().clone() for row in energy_rows], level="system"
)
hook_input.add_key(
    "forces", [row.detach().clone() for row in force_rows], level="node"
)
_ = hook_probe.run(hook_input.clone())

In [ ]:
print(f"{'model evaluation':16} {'graph':5} {'energy [eV]':>14} {'fmax [eV/Å]':>13}")
for row in history_hook.rows:
    print(
        f"{int(row['step']):<16} {int(row['graph']):<5} "
        f"{row['energy_ev']:>14.4f} {row['fmax_ev_per_a']:>13.4f}"
    )
# FIRE2 refills this list, so the probe rows must not stay in the history.
history_hook.rows.clear()

**Go deeper:** [Hooks](../04-hooks/hooks.ipynb) follows the complete lifecycle, including cleanup and failure behavior. The [official custom-hook example](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/02_custom_hook.html) builds a radial-distribution-function observer.

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 05 · Attempt a bounded relaxation with FIRE2

FIRE2 updates atomic positions to reduce the energy and forces predicted by the model. This example allows at most 16 updates and requests a maximum force below `0.05 eV/Å` for each molecule.

Two `ConvergenceHook` objects evaluate the same force criterion. The registered hook observes each graph during the run. The object passed as `convergence_hook=` controls host-side early exit when every graph meets the criterion. `ForceHistoryHook` records the energy and maximum force after each model evaluation.

The final table recomputes forces at the returned coordinates and applies the threshold directly.

In [ ]:
# ConvergenceHook tests the maximum atomic force against this eV/Å threshold.
FMAX_EV_PER_A = 0.05
# FIRE2 reaches this update cap when all graphs do not converge earlier.
MAX_STEPS = 16

In [ ]:
# One hook records per-graph status; the second can stop FIRE2 when all graphs converge.
status_hook = ConvergenceHook.from_fmax(
    FMAX_EV_PER_A, source_status=0, target_status=1
)
convergence_check = ConvergenceHook.from_fmax(FMAX_EV_PER_A)

In [ ]:
# FIRE2 owns position updates; the model and hooks own evaluations and observations.
# Its adaptive step starts at 0.01; maxstep caps atomic displacement at 0.04 Å.
fire2 = FIRE2(
    model=model,
    dt=0.01,
    maxstep=0.04,
    n_steps=MAX_STEPS,
    hooks=[*builtin_hooks, status_hook, history_hook],
    convergence_hook=convergence_check,
)

In [ ]:
{
    "optimizer": type(fire2).__name__,
    "maximum steps": fire2.n_steps,
    "fmax target [eV/Å]": FMAX_EV_PER_A,
    "registered hooks": [type(hook).__name__ for hook in fire2.hooks],
}

### Run the optimizer from a fresh batch

`fire2.run(...)` coordinates neighbor updates, model calls, hooks, convergence checks, and position updates. The input clones the model-ready batch, preserving its registered energy, force, and neighbor fields while leaving the original available for comparison.

In [ ]:
relaxation_input = hook_input.clone()
initial_positions = relaxation_input.positions.detach().clone()

In [ ]:
relaxed_batch = fire2.run(relaxation_input)

### Recompute and inspect the returned state

A final model evaluation refreshes energy and forces at the returned positions. The plots show each molecule's energy change and maximum force over the run, followed by a before-and-after projection of phenol.

Read the recomputed force table before interpreting the structure. A plausible picture does not establish convergence.

In [ ]:
final_batch = relaxed_batch.clone()
# Outside a dynamics host, neighbor preparation updates this Batch in place.
compute_neighbors(final_batch, config=model.model_config.neighbor_config)

In [ ]:
# FIRE2 returns positions; the model owns the energy and force evaluation.
final_output = model(final_batch)
final_batch["forces"] = final_output["forces"]

In [ ]:
force_norm = final_batch.forces.norm(dim=-1)
final_fmax = torch.zeros(final_batch.num_graphs, device=final_batch.device)
# batch_idx reduces atom-level force norms to one maximum per graph.
final_fmax.scatter_reduce_(
    0, final_batch.batch_idx, force_norm, reduce="amax", include_self=True
)

In [ ]:
final_energies = final_output["energy"].detach().cpu().flatten().tolist()
fmax_values = final_fmax.detach().cpu().tolist()
print(
    f"{'molecule':20} {'final energy [eV]':>17} "
    f"{'recomputed fmax [eV/Å]':>22} {'met threshold':>13}"
)
for label, energy, fmax in zip(labels, final_energies, fmax_values, strict=True):
    met = fmax <= FMAX_EV_PER_A
    print(f"{label:20} {energy:>17.4f} {fmax:>22.4f} {met!s:>13}")

In [ ]:
helpers.plot_fire2_evidence(history_hook.rows, FMAX_EV_PER_A, labels)

In [ ]:
structure_batch = final_batch.clone()
# Remove mixed-precision energy before recovering this structure-only view.
del structure_batch["energy"]
phenol_after_updates = structure_batch.get_data(1)
helpers.plot_structure_change(
    example_batch.get_data(1), phenol_after_updates, labels[1]
)

In [ ]:
snapshot_batch = snapshot_sink.read()
{
    "steps executed": fire2.step_count,
    "saved graph states": snapshot_batch.num_graphs,
    "recovered structure": labels[1],
    "maximum displacement [Å]": float(
        (final_batch.positions - initial_positions).norm(dim=1).max().detach().cpu()
    ),
}

**Scientific limit:** This run reaches the 16-update bound, and the recomputed table shows that none of the three structures meets `0.05 eV/Å`. The returned coordinates are not relaxed structures. Use a longer, validated protocol for production geometry optimization.

**Go deeper:** [BaseDynamics and FIRE2](../05-base-dynamics/base-dynamics.ipynb) covers status migration, coherent final recomputation, graph selection, and recovery. Compare it with the [official geometry-optimization example](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/02_geometry_optimization.html).

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 05 · Run a short molecular-dynamics stage

This section switches from molecules and AIMNet2 to 27 periodic argon atoms and a Lennard-Jones potential. The model uses `epsilon = 0.0104 eV`, `sigma = 3.40 Å`, and an `8.5 Å` cutoff.

The atoms have mass `39.948 amu`. Velocities are sampled for `50 K` in `sqrt(eV/amu)`, so `½mv²` is directly in eV. The center-of-mass velocity is removed before integration.

`NVE` performs five velocity-Verlet updates with `dt = 1 fs`. Model-provided hooks maintain neighbors, `WrapPeriodicHook` folds positions back into the cell, and a small observation hook records potential, kinetic, and total energy.

In [ ]:
# epsilon is the well depth [eV]; sigma and cutoff use the position unit [Å].
LJ_EPSILON_EV = 0.0104
LJ_SIGMA_A = 3.40
LJ_CUTOFF_A = 8.5
# Temperature sets the initial velocity scale; NVE does not thermostat it.
ARGON_INITIAL_TEMPERATURE_K = 50.0
lj_model = LennardJonesModelWrapper(
    epsilon=LJ_EPSILON_EV, sigma=LJ_SIGMA_A, cutoff=LJ_CUTOFF_A
).to(device)
argon_batch = helpers.build_argon_batch(
    device, DTYPE, temperature_k=ARGON_INITIAL_TEMPERATURE_K
)

In [ ]:
nve_trace = helpers.NVETraceHook()
NVE_DT_FS = 1.0  # Velocity-Verlet integration timestep, in femtoseconds.
NVE_STEPS = 5  # Short control-flow trace, not a production trajectory.
nve = NVE(model=lj_model, dt=NVE_DT_FS, n_steps=NVE_STEPS)
# Neighbor hooks run before compute; periodic wrapping runs after each update.
for hook in lj_model.make_neighbor_hooks():
    nve.register_hook(hook, stage=DynamicsStage.BEFORE_COMPUTE)
nve.register_hook(WrapPeriodicHook(stage=DynamicsStage.AFTER_POST_UPDATE))
nve.register_hook(nve_trace)

In [ ]:
# Inside NVE, registered hooks rebuild neighbors before each model evaluation.
argon_result = nve.run(argon_batch.clone())

In [ ]:
{
    "updates": nve.step_count,
    "atoms": argon_result.num_nodes,
    "time step": "1.0 fs",
    "temperature used for initialization": "50 K",
    "mass unit": "amu",
    "velocity unit": "sqrt(eV/amu)",
}

In [ ]:
print(
    f"{'post-update index':17} {'potential energy [eV]':>21} "
    f"{'kinetic energy [eV]':>19} {'total energy [eV]':>17}"
)
for row in nve_trace.rows:
    print(
        f"{int(row['step']):<17} {row['potential_ev']:>21.6f} "
        f"{row['kinetic_ev']:>19.6f} {row['total_ev']:>17.6f}"
    )

In [ ]:
helpers.plot_nve_trace(nve_trace.rows)

**Scientific limit:** Five updates show the integrator and hook sequence. They do not establish equilibration, energy conservation over a useful interval, or a scientific trajectory.

**Go deeper:** [BaseDynamics](../05-base-dynamics/base-dynamics.ipynb) explains the shared execution loop. The [official NVE example](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/04_nve_energy_conservation.html) runs long enough to calculate energy drift. [GPU pipelines and profiling](../06-gpu-pipelines-profiling/gpu-pipelines-profiling.ipynb) is in progress.

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 03 · Combine model components

This example keeps the argon batch and adds a deterministic quadratic energy term. `NativeQuadraticCorrection` is a PyTorch module with a coefficient of `1e-4 eV/Å²`. The visible `QuadraticCorrectionWrapper` declares the term's Toolkit inputs and outputs with `ModelConfig` and returns one energy per graph.

The expression `lj_model + correction_wrapper` returns a `PipelineModelWrapper`. We compare the native term, the wrapped term, and the sum produced by the composed model.

**What to notice:** The wrapper declares a small model contract. The pipeline combines components that share that contract.

In [ ]:
class QuadraticCorrectionWrapper(torch.nn.Module, BaseModelMixin):
    """Adapt one native energy term to the Toolkit model contract."""

    def __init__(self, native):
        super().__init__()
        self.native = native
        self.model_config = ModelConfig(
            outputs=frozenset({"energy"}),
            required_inputs=frozenset({"positions"}),
        )

    @property
    def embedding_shapes(self):
        return {}

    def compute_embeddings(self, data, **kwargs):
        return data

    def forward(self, data, **kwargs):
        # Map Toolkit graph boundaries to the native term's tensor arguments.
        graph_count = data.batch_size if isinstance(data, Batch) else 1
        batch_idx = data.batch_idx if isinstance(data, Batch) else torch.zeros(
            data.num_nodes, device=data.device, dtype=torch.long
        )
        return {"energy": self.native(data.positions, batch_idx, graph_count)}


# This eV/Å² coefficient makes a small deterministic system-energy term.
native_correction = helpers.NativeQuadraticCorrection(scale=1.0e-4).to(device)
correction_wrapper = QuadraticCorrectionWrapper(native_correction).to(device)
# Select a shared system-level output before additive composition.
lj_model.set_config("active_outputs", {"energy"})
composed_toy = lj_model + correction_wrapper
{
    "composed type": type(composed_toy).__name__,
    "required inputs": sorted(correction_wrapper.model_config.required_inputs),
    "available outputs": sorted(correction_wrapper.model_config.outputs),
}

In [ ]:
composition_batch = argon_result.clone()
# The standalone wrapper needs neighbors prepared before model evaluation.
compute_neighbors(composition_batch, config=lj_model.model_config.neighbor_config)

In [ ]:
native_energy = native_correction(
    composition_batch.positions, composition_batch.batch_idx, composition_batch.num_graphs
)
wrapped_energy = correction_wrapper(composition_batch)["energy"]
lj_energy = lj_model(composition_batch)["energy"]
composed_energy = composed_toy(composition_batch)["energy"]

In [ ]:
wrapper_delta = float((wrapped_energy - native_energy).abs().max().detach().cpu())
composition_delta = float(
    (composed_energy - (lj_energy + native_energy)).abs().max().detach().cpu()
)

In [ ]:
helpers.plot_wrapper_flow(composition_delta)

In [ ]:
{
    "native/wrapper max delta [eV]": wrapper_delta,
    "composed/additive max delta [eV]": composition_delta,
}

**Scientific limit:** The quadratic term depends on the coordinate origin and is deterministic code for testing the wrapper boundary. It is not a fitted interaction or an improvement to Lennard-Jones physics. Near-zero native/wrapped and manual/pipeline deltas are implementation-identity checks, not independent validation.

**Go deeper:** [Model interfaces and composition](../03-model-interfaces-composition/model-interfaces-composition.ipynb) develops adapters and pipeline groups. The [official additive-composition example](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/07_composable_model_composition.html) combines Lennard-Jones and Ewald models for a physically defined system.

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 07 · Take a few training steps

Use a supported pretrained potential when its chemistry and outputs fit the task. An external PyTorch model can enter the same workflows through a wrapper that implements `BaseModelMixin` and declares `ModelConfig`. When no suitable potential exists, `TrainingStrategy` can train a compatible wrapped model from initialized weights. When a pretrained model is close, `FineTuningStrategy` can adapt the full model or a selected parameter set.

This section uses a tiny synthetic dataset and a small model with pre-fit source weights so the fine-tuning loop runs in seconds. It demonstrates strategy configuration, optimizer updates, hooks, and loss recording. It is not a scientifically useful potential.

Each target is `energy = 0.8 × sum(positions²)`. The fixed-width MLP flattens coordinates, so it remains sensitive to atom order, rotation, translation, and the chosen origin. We reuse the first training minibatch for a before-and-after MSE check; that seen-batch check does not measure generalization.

In [ ]:
synthetic_model, training_loader, source_loss = helpers.prepare_synthetic_transfer(device)
training_example = next(iter(training_loader)).to(device)

In [ ]:
with torch.no_grad():
    seen_batch_loss_before = torch.nn.functional.mse_loss(
        synthetic_model(training_example)["energy"], training_example.energy
    ).item()

In [ ]:
{
    "model": type(synthetic_model).__name__,
    "graphs": training_example.num_graphs,
    "positions": tuple(training_example.positions.shape),
    "energy targets": tuple(training_example.energy.shape),
    "target unit": "synthetic scalar",
    "last source minibatch MSE": source_loss,
    "reused training minibatch MSE before updates": seen_batch_loss_before,
}

In [ ]:
parameters_before = {
    name: parameter.detach().clone()
    for name, parameter in synthetic_model.named_parameters()
}
pd.DataFrame(
    {
        "parameter": [name for name, _ in synthetic_model.named_parameters()],
        "shape": [tuple(parameter.shape) for _, parameter in synthetic_model.named_parameters()],
        "trainable": [parameter.requires_grad for parameter in synthetic_model.parameters()],
    }
)

### Configure the strategy and loss

`FineTuningStrategy` receives the model, optimizer recipe, loss function, training function, device, and observation hooks. `OptimizerConfig` keeps optimizer construction with the strategy, and `EnergyMSELoss` compares predicted and target system energies.

`num_steps=4` stops the run after four optimizer updates. The hook records a different minibatch at each update, so the plot uses unconnected points rather than implying a continuous learning curve.

In [ ]:
loss_history = helpers.LossHistoryHook()
LEARNING_RATE = 1.0e-3  # AdamW step size for this synthetic update check.
TRAINING_UPDATES = 4  # FineTuningStrategy stops after this many optimizer steps.
# FineTuningStrategy owns optimizer creation, loss evaluation, updates, and hooks.
strategy = FineTuningStrategy(
    models=synthetic_model,
    optimizer_configs=OptimizerConfig(
        optimizer_cls=torch.optim.AdamW,
        optimizer_kwargs={"lr": LEARNING_RATE},
    ),
    training_fn=default_training_fn,
    loss_fn=EnergyMSELoss(dtype_policy="prediction_to_target"),
    num_steps=TRAINING_UPDATES,
    devices=[device],
    hooks=[loss_history],
)

In [ ]:
# FineTuningStrategy owns optimization steps, scheduler updates, and hook calls.
strategy.run(training_loader)

In [ ]:
with torch.no_grad():
    seen_batch_loss_after = torch.nn.functional.mse_loss(
        synthetic_model(training_example)["energy"], training_example.energy
    ).item()

In [ ]:
parameter_delta = max(
    (parameter.detach() - parameters_before[name]).abs().max().item()
    for name, parameter in synthetic_model.named_parameters()
)
{
    "optimizer updates": strategy.step_count,
    "recorded minibatch losses": len(loss_history.rows),
    "reused training minibatch MSE before": seen_batch_loss_before,
    "reused training minibatch MSE after": seen_batch_loss_after,
    "largest parameter change": parameter_delta,
}

In [ ]:
print(f"{'optimizer update':16} {'minibatch MSE':>15}")
for row in loss_history.rows:
    print(f"{int(row['step']):<16} {row['loss']:>15.6f}")
helpers.plot_training_loss(loss_history.rows)

Each unconnected point is a different training minibatch, so the plot is not a monotonic learning curve. The reused minibatch is the comparable before-and-after fitting check; its decrease uses seen samples and says nothing about generalization. The parameter delta confirms that the optimizer changed the model.

**Scientific limit:** The dataset, target, and model are synthetic. Four updates demonstrate strategy configuration, optimizer execution, hooks, and loss recording. They do not measure chemical accuracy or improve the AIMNet2 model used earlier.

**Go deeper:** [Training and fine-tuning](../07-training-finetuning/training-finetuning.ipynb) is in progress. The [official fine-tuning guide](https://nvidia.github.io/nvalchemi-toolkit/userguide/finetuning.html) covers trainable parameter selection, validation, checkpointing, and restartable workflows.

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 08 · Preview domain parallelism in one process

Return to AIMNet2 and build one periodic system containing phenol and N-methylacetamide. In a distributed launch, `DomainParallel` can divide one large system across ranks while preserving the dynamics interface.

This notebook runs the same public boundaries in a single Python process: initialize `DistributedManager`, create `DomainConfig`, enter `DomainParallel`, call `partition(...)`, run one model evaluation, gather the result, and clean up.

World size one keeps every atom on rank 0, so this walkthrough does not demonstrate partitioning across ranks.

In [ ]:
nma_atoms, _ = helpers.load_molecules(("N-methylacetamide",))
domain_atoms = atoms[1].copy()
domain_partner = nma_atoms[0].copy()
# Separate the molecules by 9 Å inside a 30 Å periodic control cell.
domain_partner.positions += (9.0, 0.0, 0.0)
domain_atoms += domain_partner
domain_atoms.set_cell((30.0, 30.0, 30.0))
domain_atoms.center()
domain_atoms.set_pbc(True)

In [ ]:
domain_graph = AtomicData.from_atoms(domain_atoms, device=device)
domain_graph.add_system_property("charge", torch.zeros((1, 1), device=device))
domain_graph.add_system_property("energy", torch.zeros((1, 1), device=device))
# Forces are node-level because DomainParallel partitions atom rows.
domain_graph.add_node_property("forces", torch.zeros_like(domain_graph.positions))
domain_batch = Batch.from_data_list([domain_graph], device=device)

In [ ]:
# Match the model cutoff; zero skin adds no neighbor-list buffer.
# mesh=None and a 1 × 1 × 1 grid keep every atom on rank 0.
domain_config = DomainConfig(
    cutoff=float(model.model_config.neighbor_config.cutoff),
    skin=0.0,
    mesh=None,
    grid_dims=(1, 1, 1),
    compile=False,
)

In [ ]:
DistributedManager.initialize()
manager = DistributedManager()
world_size = manager.world_size
DOMAIN_STEPS = 1  # One model step exercises the domain control path.
# BaseDynamics hosts model evaluation and neighbor hooks inside DomainParallel.
domain_evaluator = BaseDynamics(
    model=model, n_steps=DOMAIN_STEPS, hooks=model.make_neighbor_hooks()
)

In [ ]:
try:
    with DomainParallel(
        dynamics=domain_evaluator,
        config=domain_config,
        n_steps=DOMAIN_STEPS,
        device_type=device.type,
    ) as domain:
        # partition assigns owned atoms, run evaluates them, and gather rebuilds rank 0.
        owned_batch = domain.partition(domain_batch)
        domain_result = domain.run(owned_batch, n_steps=DOMAIN_STEPS)
        gathered_result = domain.gather(domain_result, dst=0)
finally:
    DistributedManager.cleanup()

In [ ]:
helpers.plot_domain_control(world_size, int(owned_batch.positions.shape[0]))
{
    "world size": world_size,
    "input atoms": domain_batch.num_nodes,
    "rank-0 owned atoms": int(owned_batch.positions.shape[0]),
    "gathered atoms": int(gathered_result.positions.shape[0]),
    "gathered type": type(gathered_result).__name__,
}

**Scientific limit:** With a world size of one, rank 0 owns every atom. The call sequence is real, but no domain decomposition or scaling measurement occurs. Do not launch this exact cell unchanged on several ranks. A multi-rank version needs a device mesh, rank-0-only full input, derived or deliberately tuned grid dimensions, and energy/force parity checks before timing. Launch it outside the notebook with a model validated for the calculation.

**Go deeper:** [Domain decomposition](../08-domain-decomposition/domain-decomposition.ipynb) is in progress. The [official distributed simulations guide](https://nvidia.github.io/nvalchemi-toolkit/userguide/distributed.html) explains device meshes, partitioning, halo exchange, and launch requirements.

## Recap

The notebook used the same small set of interfaces in several settings:

- `AtomicData` stored one structure as tensors with explicit field ownership.
- `Batch` packed unequal graphs and kept their boundaries recoverable.
- Zarr saved graph records for later loading.
- Model wrappers declared inputs, outputs, neighbors, and devices.
- Hooks observed or protected named stages in a workflow.
- `FIRE2` and `NVE` reused the model, batch, and hook interfaces.
- `FineTuningStrategy` hosted a small optimizer run.
- `DomainParallel` exposed partition, run, and gather boundaries; in this one-process run, rank 0 retained every atom.

When a calculation fails, inspect these boundaries in order: data fields, batch ownership, model configuration, hook stage, workflow result, then distributed ownership.

## Choose what to study next

Follow the object closest to the work you plan to do:

- start with [AtomicData and Batch](../01-atomicdata-batch/atomicdata-and-batch.ipynb) for data conversion and ownership;
- use [Zarr data loading](../02-zarr-data-loading/zarr-data-loading.ipynb) for stored datasets;
- continue with [model interfaces](../03-model-interfaces-composition/model-interfaces-composition.ipynb), [hooks](../04-hooks/hooks.ipynb), or [BaseDynamics](../05-base-dynamics/base-dynamics.ipynb) for simulation software;
- [GPU pipelines and profiling](../06-gpu-pipelines-profiling/gpu-pipelines-profiling.ipynb), [training and fine-tuning](../07-training-finetuning/training-finetuning.ipynb), and [domain decomposition](../08-domain-decomposition/domain-decomposition.ipynb) are in progress.

**Go deeper:** Keep the [Toolkit user guides](https://nvidia.github.io/nvalchemi-toolkit/userguide/index.html) and [examples gallery](https://nvidia.github.io/nvalchemi-toolkit/examples/) beside you when adapting this playbook to a new model or system.

In [ ]:
assert not STORE.parent.exists()
print("Temporary Zarr store removed.")